# Практика · Навчання CNN

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає двадцять пʼять мереж — усі пʼять замірів теми плюс одне повне
> навчання. Заміряно `check_notebook.py`: **135 секунд** на процесорі без відеокарти,
> в один потік. Це нормально: навчання і є предметом теми.

Мережа в нас уже є (тема 09), дані теж (тема 10). Тут ми розбираємось із тим, **як її
навчати** — що вмикати, за чим стежити й що робити, коли пішло не так.

Що зробимо:

1. **Нормалізація входу** — замір №1: та сама мережа на `0..255` і на нормалізованих даних.
2. **Звірка з бібліотекою**: наша нормалізація по каналах проти `transforms.Normalize`.
3. **Батчнорм** — замір №2: епох до заданої точності й чи можна брати більший крок.
4. **Швидкість навчання** — замір №3: чотири значення, три режими на кривих.
5. **Розмір батча** — замір №4: час епохи проти якості.
6. **Дропаут** — замір №5: чи допомагає він у згортках, чи лише в голові.
7. **Забутий `model.eval()`** — помилка, яка не дає повідомлення про помилку.
8. **Повне навчання** один раз, із кривими для діагностики.

**Мережа не потрібна:** датасет ми малюємо самі, формулами.

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn

# зерна фіксуємо найпершими рядками: без них жодне порівняння нижче не повториться
torch.manual_seed(0)
rng = np.random.default_rng(42)

# Один потік — навмисно, з двох причин.
# 1) Відтворюваність: багатопотокові операції складають числа з рухомою комою в
#    різному порядку, і в нестійких режимах навчання це розводить результати.
#    З одним потоком твої числа збігаються з лекцією до останнього знака.
# 2) Швидкість: тензори тут крихітні (батч 32 × 8 каналів × 28×28), і накладні
#    витрати на потоки більші за виграш. Заміряно: 4 епохи — 7.2 с на одному
#    потоці проти 65.2 с на чотирьох.
torch.set_num_threads(1)

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Датасет: ті самі фігури, але в сирому діапазоні

Генератор той самий, що в темі 09, з однією відмінністю: значення пікселів лишаються
в діапазоні **0..255**, як у справжньому файлі з диска. Саме з цього діапазону починається
перший замір теми.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=2, noise=0.10):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..255."""
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо на кілька пікселів, щоб мережа не завчила одне положення
    center_y = size / 2 + rng.integers(-jitter, jitter + 1)
    center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(7, 10)

    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    # 255 замість 1.0: саме такі числа приходять із файлу, і саме вони ламають навчання
    return np.clip(image, 0, 1) * 255.0


def make_shape_dataset(count, rng, jitter=2):
    """Повертає тензори (count, 1, 28, 28) у діапазоні 0..255 і мітки (count,)."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


started = time.time()
# мала вибірка — саме для порівнянь: їх буде багато, і кожне має коштувати копійки
train_raw, train_labels = make_shape_dataset(360, rng)
test_raw, test_labels = make_shape_dataset(300, rng)
print("згенеровано за %.2f с" % (time.time() - started))
print("навчальна вибірка:", tuple(train_raw.shape), "перевірочна:", tuple(test_raw.shape))
print("діапазон значень: %.1f … %.1f" % (train_raw.min(), train_raw.max()))

## 2 · Нормалізація: середнє й стандартне відхилення по каналу

Правило одне й воно жорстке: **обидва числа рахуються по навчальній вибірці**, а до
перевірочної застосовуються ті самі. Інакше перевірочна вибірка підказує моделі, який у неї
розподіл, — і оцінка виходить завищеною.

У нас один канал, тому чисел буде по одному. Для кольорового зображення їх було б по три —
формула та сама, лише усереднення йде по осях `(0, 2, 3)`: по прикладах, по висоті й по
ширині, але **не** по каналах.

In [ ]:
# усереднюємо по прикладах, висоті й ширині — канал лишається окремим
channel_mean = train_raw.mean(dim=(0, 2, 3), keepdim=True)
channel_std = train_raw.std(dim=(0, 2, 3), keepdim=True)

print("середнє по каналу        :", channel_mean.flatten().tolist())
print("станд. відхилення по каналу:", channel_std.flatten().tolist())

# ті самі два числа йдуть і на перевірочну вибірку — саме тому вони збережені у змінних
train_norm = (train_raw - channel_mean) / channel_std
test_norm = (test_raw - channel_mean) / channel_std

print()
print("до нормалізації : mean %8.3f  std %7.3f  діапазон %.1f … %.1f"
      % (train_raw.mean(), train_raw.std(), train_raw.min(), train_raw.max()))
print("після           : mean %8.3f  std %7.3f  діапазон %.2f … %.2f"
      % (train_norm.mean(), train_norm.std(), train_norm.min(), train_norm.max()))

### Наше проти бібліотечного

`torchvision.transforms.v2.Normalize` робить рівно те саме віднімання й ділення. Перевіримо
це прямо — з однією тонкістю, про яку легко забути: `Normalize` очікує значення в
`0..1`, тому середнє й відхилення для неї треба подати в тому ж масштабі, тобто поділені
на 255.

In [ ]:
from torchvision.transforms import v2

# бібліотека працює з діапазоном 0..1, тому й статистики переводимо в нього
library_normalize = v2.Normalize(mean=[channel_mean.item() / 255.0],
                                 std=[channel_std.item() / 255.0])
from_library = library_normalize(train_raw / 255.0)

biggest_difference = (from_library - train_norm).abs().max().item()
print("найбільша розбіжність між нашою нормалізацією й бібліотечною: %.2e"
      % biggest_difference)

assert torch.allclose(from_library, train_norm, atol=1e-4), "нормалізація розійшлася!"
print("✅ збігається: transforms.Normalize — це те саме віднімання й ділення")

## 3 · Мережа й цикл навчання

Мережа — та сама, що в темі 09: три блоки `Conv → ReLU → Pool` і голова з двох
повнозвʼязних шарів. Додано два перемикачі, які нам знадобляться далі: чи ставити батчнорм
після кожної згортки й чи ставити дропаут (окремо в тілі, окремо в голові).

In [ ]:
def conv_block(in_channels, out_channels, batchnorm=False, dropout=0.0):
    """Один блок. Батчнорм стоїть МІЖ згорткою і ReLU — саме там від нього користь."""
    layers = [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)]
    if batchnorm:
        layers.append(nn.BatchNorm2d(out_channels))
    layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(2))
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)


class ShapeNet(nn.Module):
    """Тіло з трьох блоків і голова. Перемикачі — щоб порівнювати ту саму конструкцію."""

    def __init__(self, batchnorm=False, conv_dropout=0.0, head_dropout=0.0):
        super().__init__()
        self.body = nn.Sequential(
            conv_block(1, 8, batchnorm, conv_dropout),
            conv_block(8, 16, batchnorm, conv_dropout),
            conv_block(16, 32, batchnorm, conv_dropout),
        )
        head_layers = [nn.Flatten()]
        if head_dropout > 0:
            head_layers.append(nn.Dropout(head_dropout))
        head_layers += [nn.Linear(32 * 3 * 3, 64), nn.ReLU(), nn.Linear(64, 6)]
        self.head = nn.Sequential(*head_layers)

    def forward(self, x):
        return self.head(self.body(x))


plain_net = ShapeNet()
batchnorm_net = ShapeNet(batchnorm=True)
print("параметрів без батчнорму:", sum(p.numel() for p in plain_net.parameters()))
print("параметрів із батчнормом:", sum(p.numel() for p in batchnorm_net.parameters()))
print("різниця — це по два числа на канал:", 2 * (8 + 16 + 32))

Цикл навчання окремою функцією: усі порівняння нижче мусять відрізнятися **рівно однією**
річчю, а решта — включно з зерном — мусить бути однаковою. Функція повертає криві, бо саме
з кривих і читається діагноз.

Зверни увагу на два виклики всередині: `model.eval()` перед вимірюванням і `model.train()`
одразу після. Без них батчнорм і дропаут працювали б у режимі навчання під час перевірки —
і саме цю помилку ми окремо заміряємо в розділі 9.

In [ ]:
def accuracy(model, images, labels):
    """Частка правильних відповідей. eval() обовʼязковий: він перемикає батчнорм і дропаут."""
    model.eval()                                  # режим перевірки
    with torch.no_grad():
        predicted = model(images).argmax(dim=1)
    result = (predicted == labels).float().mean().item()
    model.train()                                 # повертаємось у режим навчання
    return result


def run_training(model, train_images, train_labels, test_images, test_labels,
                 epochs=6, lr=0.03, batch_size=32, optimizer_name="sgd",
                 weight_decay=0.0, seed=0, schedule=None):
    """Навчає модель і повертає криві. Зерно фіксується тут, а не зовні — щоб порядок
    перемішування був однаковим у всіх порівнюваних прогонах."""
    torch.manual_seed(seed)
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                                    weight_decay=weight_decay)
    loss_function = nn.CrossEntropyLoss()

    steps_per_epoch = math.ceil(len(train_images) / batch_size)
    total_steps = steps_per_epoch * epochs
    step = 0
    history = {"loss": [], "train_accuracy": [], "test_accuracy": [], "lr": []}

    started = time.time()
    model.train()
    for epoch in range(epochs):
        order = torch.randperm(len(train_images))
        loss_sum, seen = 0.0, 0
        for start in range(0, len(train_images), batch_size):
            batch_index = order[start:start + batch_size]
            if schedule is not None:
                # розклад міняє швидкість на кожному кроці, а не раз на епоху
                for group in optimizer.param_groups:
                    group["lr"] = schedule(step, total_steps, lr)
            optimizer.zero_grad()
            loss = loss_function(model(train_images[batch_index]),
                                 train_labels[batch_index])
            loss.backward()
            optimizer.step()
            # середня втрата по епосі чесніша за втрату останнього батча
            loss_sum += loss.item() * len(batch_index)
            seen += len(batch_index)
            step += 1
        history["lr"].append(optimizer.param_groups[0]["lr"])
        history["loss"].append(loss_sum / seen)
        history["train_accuracy"].append(accuracy(model, train_images, train_labels))
        history["test_accuracy"].append(accuracy(model, test_images, test_labels))
    history["seconds"] = time.time() - started
    return history


def show(title, history):
    """Однаковий друк для всіх замірів: втрати по епохах, точність, час."""
    print("%-26s втрати %s | тест %s | %.1f с"
          % (title,
             " ".join("%5.2f" % value for value in history["loss"]),
             " ".join("%.2f" % value for value in history["test_accuracy"]),
             history["seconds"]))


print("функції готові: run_training і show")

## 4 · Замір №1: нормалізація входу

Дві однакові мережі, однакове зерно, однакова швидкість навчання. Відрізняється **рівно
одне** — числа на вході: `0..255` проти нормалізованих.

In [ ]:
torch.manual_seed(0)
net_on_raw = ShapeNet()
raw_history = run_training(net_on_raw, train_raw, train_labels, test_raw, test_labels,
                           epochs=6, lr=0.03)
show("вхід 0..255", raw_history)

torch.manual_seed(0)
net_on_norm = ShapeNet()
norm_history = run_training(net_on_norm, train_norm, train_labels, test_norm, test_labels,
                            epochs=6, lr=0.03)
show("вхід нормалізований", norm_history)

print()
print("точність на сирому вході      : %.3f" % raw_history["test_accuracy"][-1])
print("точність на нормалізованому   : %.3f" % norm_history["test_accuracy"][-1])
print("рівень вгадування на 6 класах : %.3f" % (1 / 6))

Числа на вході множаться на ваги, а ваги при ініціалізації мають порядок десятих і сотих.
Подивимось, наскільки різні відгуки першого шару в двох випадках — це і є пояснення
результату вище одним числом.

In [ ]:
torch.manual_seed(0)
fresh = ShapeNet()                      # свіжа мережа: дивимось на стан ДО навчання
with torch.no_grad():
    response_raw = fresh.body[0][0](train_raw[:32])
    response_norm = fresh.body[0][0](train_norm[:32])

print("типова вага першої згортки (std): %.4f" % fresh.body[0][0].weight.std().item())
print()
print("відгук на сирому вході      : std %8.3f  діапазон %8.2f … %.2f"
      % (response_raw.std(), response_raw.min(), response_raw.max()))
print("відгук на нормалізованому   : std %8.3f  діапазон %8.2f … %.2f"
      % (response_norm.std(), response_norm.min(), response_norm.max()))
print()
print("у %.0f разів більший розмах ще до навчання — і саме його потім множать на швидкість"
      % (response_raw.std() / response_norm.std()))

### Розгортка по швидкості навчання

Порівняння вище зроблене при **одній** швидкості — і саме тому сирий вхід програв.
Питання цікавіше: чи є для нього своя швидкість, на якій він теж навчиться.

Проженемо три масштаби входу × чотири швидкості. Це дванадцять коротких навчань,
близько пів хвилини разом. Саме ця таблиця стоїть у лекції.


In [ ]:
train_unit = train_raw / 255.0            # третій масштаб: 0…1, найпоширеніший на практиці
test_unit = test_raw / 255.0

scales = {
    "0…255":          (train_raw, test_raw),
    "0…1":            (train_unit, test_unit),
    "нормалізований": (train_norm, test_norm),
}
learning_rates = [0.0003, 0.003, 0.03, 0.3]

print(f"{'масштаб':<16}" + "".join(f"{lr:>10}" for lr in learning_rates))
print("-" * (16 + 10 * len(learning_rates)))
sweep = {}
for scale_name, (train_x, test_x) in scales.items():
    row = []
    for lr in learning_rates:
        torch.manual_seed(0)                  # однакові початкові ваги в усіх клітинках
        model = ShapeNet()
        history = run_training(model, train_x, train_labels, test_x, test_labels,
                               epochs=6, lr=lr)
        row.append(history["test_accuracy"][-1])
    sweep[scale_name] = row
    print(f"{scale_name:<16}" + "".join(f"{value:>10.3f}" for value in row))

print()
print("рівень вгадування на 6 класах: %.3f" % (1 / 6))
for scale_name, row in sweep.items():
    best = max(range(len(row)), key=lambda i: row[i])
    print(f"  {scale_name:<16} найкраще {row[best]:.3f} при швидкості {learning_rates[best]}")


## 5 · Замір №2: батчнорм

Тепер та сама мережа з `nn.BatchNorm2d` після кожної згортки й без нього. Два питання:
скільки епох до заданої точності — і чи можна після батчнорму брати **більшу** швидкість
навчання. Друге питання практично важливіше за перше.

Швидкості беремо дві: звичну 0.02 і 0.15 — ту, на якій у лекції мережа без батчнорму
вже розвалюється.

In [ ]:
def epochs_to_reach(history, target):
    """Номер першої епохи, на якій точність на перевірці досягла цілі."""
    for number, value in enumerate(history["test_accuracy"], start=1):
        if value >= target:
            return number
    return None


TARGET = 0.90
batchnorm_results = {}

for lr in (0.02, 0.15):
    for use_batchnorm in (False, True):
        torch.manual_seed(0)
        model = ShapeNet(batchnorm=use_batchnorm)
        history = run_training(model, train_norm, train_labels, test_norm, test_labels,
                               epochs=8, lr=lr)
        batchnorm_results[(lr, use_batchnorm)] = history
        show("lr=%-5g батчнорм %s" % (lr, "є " if use_batchnorm else "нема"), history)

print()
print("%-8s %-10s %14s %10s" % ("lr", "батчнорм", "епох до 0.90", "фінал"))
print("-" * 46)
for (lr, use_batchnorm), history in batchnorm_results.items():
    reached = epochs_to_reach(history, TARGET)
    print("%-8g %-10s %14s %10.3f"
          % (lr, "є" if use_batchnorm else "нема",
             reached if reached else "не дійшла", history["test_accuracy"][-1]))

## 6 · Замір №3: швидкість навчання

Чотири значення від замалого до завеликого. На кривих втрат мусять бути видні всі три
режими: повзе, збігається, розходиться.

In [ ]:
learning_rate_results = {}
for lr in (0.0003, 0.03, 0.3, 1.0):
    torch.manual_seed(0)
    model = ShapeNet()
    history = run_training(model, train_norm, train_labels, test_norm, test_labels,
                           epochs=8, lr=lr)
    learning_rate_results[lr] = history
    show("lr = %g" % lr, history)

print()
print("%-10s %10s %10s %10s" % ("lr", "втрата 1", "втрата 8", "фінал"))
print("-" * 44)
for lr, history in learning_rate_results.items():
    print("%-10g %10.2f %10.2f %10.3f"
          % (lr, history["loss"][0], history["loss"][-1],
             history["test_accuracy"][-1]))

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for lr, history in learning_rate_results.items():
    epochs_axis = range(1, len(history["loss"]) + 1)
    axes[0].plot(epochs_axis, history["loss"], marker="o", label="lr = %g" % lr)
    axes[1].plot(epochs_axis, history["test_accuracy"], marker="o", label="lr = %g" % lr)
axes[0].set_yscale("log")
axes[0].set_title("втрата на навчанні (лог. шкала)")
axes[0].set_xlabel("епоха")
axes[1].set_title("точність на перевірці")
axes[1].set_xlabel("епоха")
axes[1].axhline(1 / 6, linestyle="--", color="gray", linewidth=1)
for axis in axes:
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("пунктир на правому графіку — рівень вгадування 1/6")

## 7 · Замір №4: розмір батча

Три значення. Дивимось на дві речі одразу: скільки коштує епоха й яка виходить якість.
Кроків за епоху при більшому батчі менше — і саме це, а не сам батч, зазвичай і псує
результат при однаковій кількості епох.

In [ ]:
print("%-8s %10s %14s %10s" % ("батч", "кроків", "час епохи", "фінал"))
print("-" * 46)
for batch_size in (8, 32, 128):
    torch.manual_seed(0)
    model = ShapeNet()
    history = run_training(model, train_norm, train_labels, test_norm, test_labels,
                           epochs=8, lr=0.03, batch_size=batch_size)
    steps_per_epoch = math.ceil(len(train_norm) / batch_size)
    print("%-8d %10d %11.2f с %10.3f"
          % (batch_size, steps_per_epoch, history["seconds"] / 8,
             history["test_accuracy"][-1]))

## 8 · Замір №5: дропаут у згортковій мережі

Питання, на яке відповідь заведено давати з памʼяті: чи допомагає дропаут у згортках.
Заміряємо — але спершу треба, щоб перенавчання взагалі було.

На нашій легкій задачі його немає: мережа доходить до 1.000 і на навчальній, і на
перевірочній вибірці, регуляризувати нічого. Тому робимо задачу **важкою**: беремо
всього 60 прикладів (по десять на клас) і піднімаємо шум із 0.10 до 0.45.

In [ ]:
def make_hard_dataset(count, rng, noise):
    """Той самий генератор, але з сильним шумом — задача стає по-справжньому важкою."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6
        images[i, 0] = draw_shape(kind, rng, noise=noise)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


hard_rng = np.random.default_rng(7)
hard_train, hard_train_labels = make_hard_dataset(60, hard_rng, 0.45)
hard_test, hard_test_labels = make_hard_dataset(300, hard_rng, 0.45)

# статистики знову з навчальної вибірки — правило те саме, що в розділі 2
hard_mean, hard_std = hard_train.mean(), hard_train.std()
hard_train = (hard_train - hard_mean) / hard_std
hard_test = (hard_test - hard_mean) / hard_std

print("важка вибірка:", tuple(hard_train.shape), "— по десять прикладів на клас")
print()
print("%-20s %8s %8s %8s" % ("дропаут", "навч.", "тест", "розрив"))
print("-" * 48)
for label, conv_dropout, head_dropout in (("немає", 0.0, 0.0),
                                          ("у згортках 0.25", 0.25, 0.0),
                                          ("у голові 0.5", 0.0, 0.5),
                                          ("у голові 0.3", 0.0, 0.3)):
    torch.manual_seed(0)
    model = ShapeNet(conv_dropout=conv_dropout, head_dropout=head_dropout)
    history = run_training(model, hard_train, hard_train_labels,
                           hard_test, hard_test_labels,
                           epochs=25, lr=0.003, optimizer_name="adam")
    print("%-20s %8.3f %8.3f %8.3f"
          % (label, history["train_accuracy"][-1], history["test_accuracy"][-1],
             history["train_accuracy"][-1] - history["test_accuracy"][-1]))

## 9 · Забутий `model.eval()`

Помилка, яка не дає повідомлення про помилку. Візьмемо навчену мережу з батчнормом і
поміряємо ту саму точність на тих самих даних двічі: у режимі `eval` і в режимі `train`.

Різниця залежить від розміру батча, тому міряємо на кількох. У режимі `train` батчнорм
рахує середнє **по поточному батчу** — і при батчі 1 йому нема з чого рахувати.

In [ ]:
torch.manual_seed(0)
trained_batchnorm = ShapeNet(batchnorm=True)
batchnorm_history = run_training(trained_batchnorm, train_norm, train_labels,
                                 test_norm, test_labels, epochs=8, lr=0.05)


def accuracy_in_mode(model, images, labels, batch_size, use_train_mode):
    """Та сама точність, але з явним вибором режиму й розміру батча."""
    model.train() if use_train_mode else model.eval()
    correct = 0
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            predicted = model(images[start:start + batch_size]).argmax(dim=1)
            correct += (predicted == labels[start:start + batch_size]).sum().item()
    model.eval()
    return correct / len(images)


print("мережа навчена, точність наприкінці навчання: %.3f"
      % batchnorm_history["test_accuracy"][-1])

# Тонкість, через яку легко зіпсувати сам замір: прохід у режимі train() ОНОВЛЮЄ
# накопичені статистики батчнорму — навіть під no_grad. Тому спершу міряємо всі
# розміри батча в режимі eval(), і лише потім усі в режимі train().
BATCH_SIZES = (1, 4, 32, 300)
correct_mode = [accuracy_in_mode(trained_batchnorm, test_norm, test_labels, size, False)
                for size in BATCH_SIZES]
wrong_mode = [accuracy_in_mode(trained_batchnorm, test_norm, test_labels, size, True)
              for size in BATCH_SIZES]

print()
print("%-10s %12s %12s %10s" % ("батч", "eval()", "train()", "різниця"))
print("-" * 48)
for number, batch_size in enumerate(BATCH_SIZES):
    print("%-10d %12.3f %12.3f %10.3f"
          % (batch_size, correct_mode[number], wrong_mode[number],
             wrong_mode[number] - correct_mode[number]))

Тепер найважливіше: чи ламається так само мережа **без** батчнорму. Якщо ні — значить,
винен не режим сам по собі, а конкретні шари, які в ньому поводяться інакше.

In [ ]:
torch.manual_seed(0)
plain_trained = ShapeNet()
plain_history = run_training(plain_trained, train_norm, train_labels,
                             test_norm, test_labels, epochs=8, lr=0.05)

print("мережа без батчнорму навчена, точність %.3f" % plain_history["test_accuracy"][-1])
print()
print("без батчнорму й дропауту:")
print("  eval()  %.3f" % accuracy_in_mode(plain_trained, test_norm, test_labels, 1, False))
print("  train() %.3f" % accuracy_in_mode(plain_trained, test_norm, test_labels, 1, True))
print()
print("Різниці немає: у мережі без батчнорму й дропауту режим не міняє нічого.")
print("Саме тому помилку так довго не помічають — вона зʼявляється лише разом із ними.")

## 10 · Повне навчання один раз

Усі порівняння вище йшли на 360 прикладах і кількох епохах — саме тому вони й коштували
секунди. Тепер одне повноцінне навчання на 1200 прикладах із усім, що ми зʼясували:
нормалізований вхід, батчнорм, розумна швидкість, косинусний спад.

In [ ]:
big_train, big_train_labels = make_shape_dataset(1200, rng)
big_test, big_test_labels = make_shape_dataset(600, rng)

# статистики знову з НАВЧАЛЬНОЇ вибірки — нові дані, нові числа
big_mean = big_train.mean(dim=(0, 2, 3), keepdim=True)
big_std = big_train.std(dim=(0, 2, 3), keepdim=True)
big_train = (big_train - big_mean) / big_std
big_test = (big_test - big_mean) / big_std


def cosine_schedule(step, total_steps, base_lr):
    """Швидкість плавно спадає від base_lr до нуля за косинусом."""
    return base_lr * 0.5 * (1 + math.cos(math.pi * step / total_steps))


torch.manual_seed(0)
final_net = ShapeNet(batchnorm=True)
final_history = run_training(final_net, big_train, big_train_labels,
                             big_test, big_test_labels,
                             epochs=10, lr=0.1, schedule=cosine_schedule)

print("%-8s %10s %12s %12s" % ("епоха", "втрата", "навчання", "перевірка"))
print("-" * 46)
for number in range(len(final_history["loss"])):
    print("%-8d %10.3f %12.3f %12.3f"
          % (number + 1, final_history["loss"][number],
             final_history["train_accuracy"][number],
             final_history["test_accuracy"][number]))
print("-" * 46)
print("навчання зайняло %.1f с" % final_history["seconds"])

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
epochs_axis = range(1, len(final_history["loss"]) + 1)
axes[0].plot(epochs_axis, final_history["loss"], marker="o", color="crimson")
axes[0].set_title("втрата на навчанні")
axes[0].set_xlabel("епоха")
axes[1].plot(epochs_axis, final_history["train_accuracy"], marker="o", label="навчання")
axes[1].plot(epochs_axis, final_history["test_accuracy"], marker="s", label="перевірка")
axes[1].set_title("точність")
axes[1].set_xlabel("епоха")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

gap = final_history["train_accuracy"][-1] - final_history["test_accuracy"][-1]
print("розрив між навчанням і перевіркою наприкінці: %.3f" % gap)
print("Це і є та пара кривих, за якою в розділі 8 лекції ставиться діагноз.")

## 11 · Відтворюваність: чому одного зерна мало

Запустимо те саме навчання **пʼять разів із різними зернами** — і подивимось, наскільки
плаває результат. Це число потрібне щоразу, коли порівнюєш дві моделі: різниця, менша за
розкид від зерна, нічого не означає.

Міряємо на важкій вибірці з розділу 8: на легкій задача надто проста, там усі зерна
впираються в 1.000 і розкид виходить нульовим.

In [ ]:
# міряємо на ВАЖКІЙ версії задачі — на легкій усі зерна впираються в 1.000
# і розкид виходить нульовим, тобто нічого не показує
seed_results = []
for seed in (0, 1, 2, 3, 4):
    torch.manual_seed(seed)
    model = ShapeNet()
    history = run_training(model, hard_train, hard_train_labels,
                           hard_test, hard_test_labels,
                           epochs=25, lr=0.003, optimizer_name="adam", seed=seed)
    seed_results.append(history["test_accuracy"][-1])
    print("зерно %d → точність %.3f" % (seed, history["test_accuracy"][-1]))

spread = max(seed_results) - min(seed_results)
print()
print("розкид від самого лише зерна: %.3f" % spread)
print("Різниця між двома моделями, менша за це число, — ще не різниця.")
print("Порівняй із таблицею дропауту вище: найкращий варіант там випередив")
print("«без дропауту» на кілька тисячних — тобто ні на скільки.")

## Завдання

### 🟢 Рівень 1

Повтори замір №1 (нормалізація), але замість `SGD` візьми `Adam` із `lr=0.003`. Чи
рятує адаптивний оптимізатор сирий вхід `0..255`? Дай відповідь двома числами.

### 🟡 Рівень 2

Додай до заміру №2 третє значення швидкості навчання — таке, при якому мережа **без**
батчнорму розходиться, а з батчнормом ще вчиться. Знайди його перебором і покажи обидві
криві втрат на одному графіку.

### 🔴 Рівень 3

Візьми зламане навчання з `homework.html` (рівень 3), постав діагноз за кривими й полагодь
його. Кожне виправлення роби окремо й показуй, скільки додало саме воно.